# Chronos-2 기반 시계열 이상 탐지 (변수별, multivariate)

**아이디어**: 품질 변수와 POLYCOM 운전시간을 제외한 공정 변수 38개를 Chronos-2에 함께 입력하고, 과거 168시간으로 1시간 뒤를 예측합니다. 실제값이 예측 분포의 1~99% 구간 밖이면 이상으로 표시합니다.

기존 `chronos-bolt-small`(univariate, 변수 하나만 보고 그 변수를 예측)과 달리, **Chronos-2는 여러 변수를 함께 입력하면 서로의 패턴을 참고해서 예측**합니다 — 예: FEED량은 정상인데 그에 맞춰 같이 움직여야 할 온도가 안 움직이는 경우도 잡을 수 있습니다.

**사용 방법**
1. `CONFIG` 셀에서 `EXCEL_PATH`/`SHEET_NAME`이 맞는지 확인 (현재 2CM 운전 데이터로 설정됨)
2. 전처리는 GitHub Cement `review_v2`처럼 물리 한계 위반값만 NaN 처리하고, IQR/주변 중앙값 치환은 하지 않음
3. `QUICK_TEST_ROWS`로 소규모 확인 후 `None`으로 바꿔 전체 재실행
4. Full 파인튜닝은 시간순 70/15/15 분할 셀부터 순서대로 실행

**환경 확인 결과**: `amazon/chronos-2` 모델이 이 Mac에서 MPS(Apple GPU)로 정상 로드/추론되는 것 확인했습니다. Apache-2.0 라이선스 오픈소스 모델이라 비용 없이 사용 가능합니다.

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from chronos import BaseChronosPipeline

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (12, 4)

# macOS 기본 폰트(DejaVu Sans)에는 한글 글리프가 없어서 컬럼명의 한글이 깨져(□) 보이는 문제 방지
plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False  # AppleGothic엔 유니코드 마이너스(−) 글리프가 없어서 꺼줌


In [ ]:
# ===================== CONFIG: GitHub Cement review_v2 방식 =====================
from pathlib import Path

from cement_chronos_review_v2 import (
    DOWNTIME_COL, PROTOCOL_VERSION, QUALITY_COLS,
    chronological_split, equipment_frame, file_sha256,
    load_preprocessed_equipment, loss_history_from_callback,
    make_loss_history_callback, rolling_forecast_multivariate as rolling_forecast_review_v2,
    runtime_versions, to_chronos_inputs, validation_windows,
)

EXCEL_PATH = "운전데이터_통합_2CM_3CM_4CM.xlsx"
SHEET_NAME = "운전데이터_통합"
HEADER_ROW = 0
EQUIPMENT_IDS = ["2CM", "3CM", "4CM"]

# 품질 2개와 POLYCOM 운전시간은 예측 대상에서 제외한다.
# 품질/비고/품종은 전처리 후 모델 입력에서 빠지고, 운전시간은 정지 판정과 past covariate로만 남는다.
PREDICT_EXCLUDE_COLS = [DOWNTIME_COL, *QUALITY_COLS]
GLITCH_VARIANTS = []                 # 기존 중앙값 글리치 분기는 실행하지 않음

CONTEXT_LENGTH = 168                # 과거 1주일
PREDICTION_LENGTH = 1               # 요청대로 1시간 뒤 하나만 예측
STRIDE = 1
assert PREDICTION_LENGTH == 1, "현재 review_v2 파이프라인은 prediction_length=1로 고정합니다."

ANOMALY_QUANTILE_LOW = 0.01
ANOMALY_QUANTILE_HIGH = 0.99
SEVERITY_EPS = 1e-3

# 통계적 IQR 제거와 ±3시간 중앙값 치환은 사용하지 않는다.
# 명백한 물리/설비 한계 위반값만 NaN으로 마스킹하며 상세 기준은 cement_chronos_review_v2.py에 있다.
PREPROCESSED_DIR = Path("data/processed")
PREPROCESSED_CSV = PREPROCESSED_DIR / "process_timeseries_review_v2.csv"

MODEL_ID = "amazon/chronos-2"
BATCH_WINDOWS = 128
QUICK_TEST_ROWS = None              # None이면 전체 기간

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print(f"Selected device: {DEVICE}, protocol={PROTOCOL_VERSION}, prediction_length={PREDICTION_LENGTH}")


In [ ]:
# ===================== [레거시 결과 조회] 기존 glitch raw/cleaned CSV =====================
# 아래 셀 3~7은 이전 실행 결과를 재현하기 위해 남겨둔 조회 전용 구간입니다. 새 review_v2 결과가 아닙니다.
# 이미 저장된 chronos_anomaly_results_{run}.csv 6개만 있으면 아래 분석(설비별 비교, ACF vs R² 비교,
# 정확도 시각화, ACF 분석)을 전부 할 수 있습니다. Chronos 모델 로드나 메인 파이프라인(load_data/
# 글리치정리/rolling_forecast)을 다시 돌릴 필요가 전혀 없어요 -- CSV에 필요한 값(actual/pred_median/
# pred_low/pred_high/is_anomaly/severity 등)이 이미 다 저장되어 있습니다.
# 이 셀 실행 전에 "import"와 "CONFIG" 셀만 실행되어 있으면 됩니다 (모델 로드/메인 파이프라인 셀은 건너뛰어도 됨).
import os

RUN_KEYS = ["2CM__glitch_cleaned", "2CM__glitch_raw",
            "3CM__glitch_cleaned", "3CM__glitch_raw",
            "4CM__glitch_cleaned", "4CM__glitch_raw"]


def load_run_csv(run_key):
    df = pd.read_csv(f"chronos_anomaly_results_{run_key}.csv")
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df


def compute_accuracy_summary_csv(df, mape_min_abs=1e-6):
    """compute_accuracy_summary와 로직은 동일하지만, CSV엔 idx가 없어서 timestamp로 정렬함
    (reindex된 1시간 grid라 timestamp 순 정렬 = idx 순 정렬과 동일)."""
    rows = []
    for col, sub in df.groupby("variable"):
        sub = sub.sort_values("timestamp").reset_index(drop=True)
        actual = sub["actual"].to_numpy()
        pred = sub["pred_median"].to_numpy()
        scored = sub["excluded_reason"].isna().to_numpy()

        naive_pred = np.roll(actual, 1)
        naive_pred[0] = np.nan
        valid = scored & ~np.isnan(naive_pred)
        n_scored = int(valid.sum())
        if n_scored < 2:
            continue

        actual_v = actual[valid]
        err = actual_v - pred[valid]
        naive_err = actual_v - naive_pred[valid]
        mae = np.mean(np.abs(err))
        naive_mae = np.mean(np.abs(naive_err))
        rmse = np.sqrt(np.mean(err ** 2))

        mape_mask = np.abs(actual_v) > mape_min_abs
        mape_n = int(mape_mask.sum())
        mape_skipped = int((~mape_mask).sum())
        mape = np.mean(np.abs(err[mape_mask] / actual_v[mape_mask])) * 100 if mape_n > 0 else np.nan

        low = sub["pred_low"].to_numpy()[valid]
        high = sub["pred_high"].to_numpy()[valid]
        coverage = ((actual_v >= low) & (actual_v <= high)).mean() * 100

        ss_res = np.sum(err ** 2)
        ss_tot = np.sum((actual_v - actual_v.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

        rows.append(dict(
            variable=col, label=col.replace("\n", " "),
            mae=mae, rmse=rmse, mape=mape, mape_n=mape_n, mape_skipped=mape_skipped,
            naive_mae=naive_mae, mae_vs_naive=mae / naive_mae if naive_mae > 0 else np.nan,
            r2=r2, coverage=coverage, n_scored=n_scored,
        ))
    return pd.DataFrame(rows)


print("CSV 로딩 중...")
csv_runs = {rk: load_run_csv(rk) for rk in RUN_KEYS}
csv_acc = {rk: compute_accuracy_summary_csv(df) for rk, df in csv_runs.items()}
print(f"{len(csv_runs)}개 run 로딩 완료: {list(csv_runs.keys())}")


In [ ]:
# ===================== [빠른 경로] 설비별(+글리치 처리 여부별) 비교 요약 (CSV 기반) =====================
compare_rows = []
for run_key, df in csv_runs.items():
    equipment_id, variant = run_key.split("__")
    acc = csv_acc[run_key]
    scored = df[df["excluded_reason"].isna()]
    compare_rows.append(dict(
        run=run_key, 설비=equipment_id, 글리치정리=(variant == "glitch_cleaned"),
        변수수=df["variable"].nunique(),
        스코어링대상=len(scored),
        이상탐지수=int(scored["is_anomaly"].sum()),
        이상비율=scored["is_anomaly"].mean() * 100,
        평균_R2=acc["r2"].mean(), 중앙값_R2=acc["r2"].median(),
        평균_coverage=acc["coverage"].mean(), 평균_MAE=acc["mae"].mean(),
    ))
equipment_comparison = pd.DataFrame(compare_rows).set_index("run")
print(equipment_comparison.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].plot(equipment_comparison.index, equipment_comparison["이상비율"], marker="o", color="tab:blue", linewidth=1.5)
axes[0].set_title("run별 이상 비율 (%)")
axes[1].plot(equipment_comparison.index, equipment_comparison["평균_R2"], marker="o", color="tab:green", linewidth=1.5)
axes[1].set_title("run별 평균 R²")
axes[2].plot(equipment_comparison.index, equipment_comparison["평균_coverage"], marker="o", color="tab:purple", linewidth=1.5)
axes[2].axhline((ANOMALY_QUANTILE_HIGH - ANOMALY_QUANTILE_LOW) * 100, color="black", linestyle="--", linewidth=1)
axes[2].set_title("run별 평균 coverage (%)")
for ax in axes:
    ax.tick_params(axis="x", labelrotation=30, labelsize=8)
plt.tight_layout()
plt.show()

equipment_comparison


In [ ]:
# ===================== [빠른 경로] 자기상관(ACF) vs R²(글리치 정리 전/후) 비교 (CSV 기반) =====================
def _acf1(x):
    a, b = x[1:], x[:-1]
    mask = ~np.isnan(a) & ~np.isnan(b)
    if mask.sum() < 10:
        return np.nan
    return np.corrcoef(a[mask], b[mask])[0, 1]


def plot_acf_vs_r2_comparison_csv(equipment_id, save_path=None):
    df_clean = csv_runs[f"{equipment_id}__glitch_cleaned"]
    acc_raw = csv_acc[f"{equipment_id}__glitch_raw"].set_index("variable")["r2"]
    acc_clean = csv_acc[f"{equipment_id}__glitch_cleaned"].set_index("variable")["r2"]

    rows = []
    for col, sub in df_clean.groupby("variable"):
        sub = sub.sort_values("timestamp")
        acf1 = _acf1(sub["actual"].to_numpy(dtype=np.float64))
        rows.append(dict(variable=col, label=col.replace("\n", " "), acf1=acf1,
                          r2_raw=acc_raw.get(col, np.nan), r2_cleaned=acc_clean.get(col, np.nan)))
    comp = pd.DataFrame(rows).dropna(subset=["acf1"]).sort_values("acf1", ascending=False).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(max(10, len(comp) * 0.35), 5))
    x = range(len(comp))
    ax.plot(x, comp["acf1"], marker="o", markersize=4, color="dimgray", linewidth=1.6, label="자기상관 ACF(lag-1)")
    ax.plot(x, comp["r2_raw"], marker="o", markersize=4, color="tab:red", linewidth=1.2,
            linestyle="--", label="R² (글리치 미정리)")
    ax.plot(x, comp["r2_cleaned"], marker="o", markersize=4, color="tab:blue", linewidth=1.2,
            label="R² (글리치 정리)")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axhline(0.5, color="gray", linewidth=0.8, linestyle=":")
    ax.set_xticks(list(x))
    ax.set_xticklabels(comp["label"], rotation=90, fontsize=7)
    ax.set_title(f"[{equipment_id}] 자기상관(ACF) vs R²(글리치 정리 전/후) -- 변수는 ACF 높은 순 정렬")
    ax.legend(loc="upper right", fontsize=9)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=120, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()

    valid = comp.dropna(subset=["r2_cleaned"])
    rho_clean = np.corrcoef(valid["acf1"].rank(), valid["r2_cleaned"].rank())[0, 1]
    rho_raw = np.corrcoef(valid["acf1"].rank(), valid["r2_raw"].rank())[0, 1]
    print(f"[{equipment_id}] Spearman(ACF, R²_raw)={rho_raw:.3f}, Spearman(ACF, R²_cleaned)={rho_clean:.3f}")
    return comp


os.makedirs("image", exist_ok=True)
for _eid in ["2CM", "3CM", "4CM"]:
    plot_acf_vs_r2_comparison_csv(_eid, save_path=os.path.join("image", f"{_eid}_acf_vs_r2.png"))
    print(f"저장 완료: image/{_eid}_acf_vs_r2.png")


In [ ]:
# ===================== [빠른 경로] 예측 정확도 시각화 (CSV 기반, run 선택) =====================
# CSV_CURRENT_RUN만 바꿔서 다시 실행하면 다른 run(설비 x 글리치처리 여부)을 볼 수 있습니다.
CSV_CURRENT_RUN = "2CM__glitch_cleaned"  # 아래 단일 조회용 (원하는 run 하나만 보고 싶을 때 사용)


def plot_accuracy_summary_csv(run_key, outlier_rmse_multiplier=20):
    acc = csv_acc[run_key].dropna(subset=["r2"])
    target_coverage = (ANOMALY_QUANTILE_HIGH - ANOMALY_QUANTILE_LOW) * 100

    median_rmse = acc["rmse"].median()
    is_extreme = acc["rmse"] > median_rmse * outlier_rmse_multiplier
    acc_extreme = acc[is_extreme]
    acc_main = acc[~is_extreme]
    acc_sorted = acc_main.sort_values("r2", ascending=False).reset_index(drop=True)
    x = range(len(acc_sorted))

    fig, axes = plt.subplots(4, 1, figsize=(max(10, len(acc_sorted) * 0.35), 14), sharex=True)
    axes[0].plot(x, acc_sorted["r2"], marker="o", markersize=4, color="tab:green", linewidth=1.2)
    axes[0].axhline(0, color="black", linewidth=0.8)
    axes[0].axhline(0.5, color="gray", linewidth=0.8, linestyle="--")
    axes[0].set_title(f"[{run_key}] R² (설명력)")

    axes[1].plot(x, acc_sorted["coverage"], marker="o", markersize=4, color="tab:purple", linewidth=1.2)
    axes[1].axhline(target_coverage, color="black", linewidth=1, linestyle="--")
    axes[1].set_title(f"예측구간 커버리지 % (점선={target_coverage:.0f}%)")

    axes[2].plot(x, acc_sorted["mae"], marker="o", markersize=4, color="tab:blue", linewidth=1.2)
    axes[2].set_title("MAE (실제 단위)")

    axes[3].plot(x, acc_sorted["mape"], marker="o", markersize=4, color="tab:orange", linewidth=1.2)
    axes[3].set_title("MAPE (%)")

    for ax in axes:
        ax.set_xticks(list(x))
        ax.set_xticklabels(acc_sorted["label"], rotation=90, fontsize=7)
        ax.tick_params(axis="x", labelbottom=True)
        ax.tick_params(axis="y", labelsize=8)
    plt.tight_layout()
    plt.show()

    print(f"[{run_key}] 평균(극단치 {len(acc_extreme)}개 제외, {len(acc_main)}개 기준): "
          f"R²={acc_main['r2'].mean():.3f}, coverage={acc_main['coverage'].mean():.2f}%, "
          f"MAE={acc_main['mae'].mean():.3f}, MAPE={acc_main['mape'].mean():.2f}%")
    print(f"R² >= 0.5인 변수: {(acc_main['r2'] >= 0.5).sum()}/{len(acc_main)}")
    if len(acc_extreme):
        print(f"\n[제외된 극단치 변수 {len(acc_extreme)}개]:")
        display(acc_extreme[["label", "r2", "coverage", "mae", "mape", "n_scored"]])
    return acc_sorted


# 설비 3개 x 글리치 정리/미정리 2개 = 총 6개 run을 전부 순서대로 보여줍니다.
for _run_key in RUN_KEYS:
    _ = plot_accuracy_summary_csv(_run_key)


In [ ]:
# ===================== [빠른 경로] 변수별 자기상관(ACF) 분석 (CSV 기반, 설비별) =====================
# ACF는 글리치 정리 여부와 무관합니다 (정리는 컨텍스트에만 적용되고, 채점용 actual 값 자체는
# raw 그대로라서 cleaned/raw 버전의 ACF가 완전히 동일함) -- 그래서 설비(2CM/3CM/4CM) 3개만 봅니다.

def compute_acf_full(x, max_lag):
    acf = np.full(max_lag + 1, np.nan)
    for k in range(max_lag + 1):
        if k == 0:
            acf[k] = 1.0
            continue
        a, b = x[k:], x[:-k]
        mask = ~np.isnan(a) & ~np.isnan(b)
        if mask.sum() < 10:
            continue
        acf[k] = np.corrcoef(a[mask], b[mask])[0, 1]
    return acf


def plot_acf_overview_csv(acf_series, title_prefix=""):
    s = acf_series.dropna().sort_values(ascending=False)
    labels = [c.replace("\n", " ") for c in s.index]
    fig, ax = plt.subplots(figsize=(max(10, len(s) * 0.35), 4))
    ax.plot(range(len(s)), s.values, marker="o", markersize=4, color="teal", linewidth=1.2)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axhline(0.5, color="gray", linewidth=0.8, linestyle="--")
    ax.set_xticks(range(len(s)))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_title(f"{title_prefix} 변수별 lag-1 자기상관(ACF)")
    plt.tight_layout()
    plt.show()


def plot_acf_detail_csv(acf_by_var, col, n, max_lag, label_prefix=""):
    label = col.replace("\n", " ")
    acf = acf_by_var[col]
    ci = 1.96 / np.sqrt(n)
    fig, ax = plt.subplots(figsize=(12, 3.5))
    ax.bar(range(max_lag + 1), acf, width=0.8, color="tab:blue")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axhline(ci, color="gray", linestyle="--", linewidth=0.8)
    ax.axhline(-ci, color="gray", linestyle="--", linewidth=0.8)
    for k in (24, 168, 336):
        if k <= max_lag:
            ax.axvline(k, color="lightgray", linewidth=1, zorder=0)
    ax.set_title(f"{label_prefix}{label} -- ACF(lag 0~{max_lag}h)")
    plt.tight_layout()
    plt.show()


ACF_MAX_LAG = 336
acf_lag1_by_equipment = {}

for _eid in ["2CM", "3CM", "4CM"]:
    _df_for_acf = csv_runs[f"{_eid}__glitch_cleaned"]  # cleaned/raw 동일하므로 하나만 사용
    _acf_by_var_csv = {}
    for col, sub in _df_for_acf.groupby("variable"):
        sub = sub.sort_values("timestamp")
        _acf_by_var_csv[col] = compute_acf_full(sub["actual"].to_numpy(dtype=np.float64), ACF_MAX_LAG)

    acf_lag1_csv = pd.Series({c: v[1] for c, v in _acf_by_var_csv.items()}, name="acf_lag1")
    acf_lag1_by_equipment[_eid] = acf_lag1_csv
    plot_acf_overview_csv(acf_lag1_csv, title_prefix=f"[{_eid}]")

    # 설비별로 R² 제일 높은 변수 2개 vs 제일 낮은 변수 2개의 ACF 구조 비교
    # (R²는 글리치정리 버전 기준으로 정렬 -- 정리한 게 대표성이 더 좋아서)
    _acc_for_acf = csv_acc[f"{_eid}__glitch_cleaned"]
    _top = _acc_for_acf.sort_values("r2", ascending=False)["variable"].head(2).tolist()
    _bottom = _acc_for_acf.sort_values("r2", ascending=True)["variable"].head(2).tolist()

    print(f"\n[{_eid}] R² 높은 변수(예측 잘 됨) vs 낮은 변수(예측 잘 안 됨)의 자기상관 구조 비교:")
    for _col in _top + _bottom:
        _n = _df_for_acf[_df_for_acf["variable"] == _col]["actual"].notna().sum()
        plot_acf_detail_csv(_acf_by_var_csv, _col, _n, ACF_MAX_LAG, label_prefix=f"[{_eid}] ")


In [ ]:
# ===================== 데이터 로드: GitHub review_v2 전처리 =====================
def load_data(equipment_id):
    """물리 기준 마스킹 -> 내수 품종 필터 -> 숫자 변환 -> 1시간 grid 재색인."""
    return load_preprocessed_equipment(
        EXCEL_PATH, SHEET_NAME, equipment_id, header_row=HEADER_ROW,
    )


In [ ]:
# ===================== 이상치 제거 원칙 =====================
# 기존 ±3시간 중앙값/Hampel 유사 글리치 탐지는 사용하지 않는다.
# GitHub Cement_code_fin의 canonical 방식처럼 명백한 물리/설비 한계 위반값만 NaN 처리한다.
# 따라서 값을 주변 중앙값으로 대체하지 않으며, 통계적 IQR 제거도 수행하지 않는다.
print("전처리: fixed physical limits -> NaN, IQR/median replacement 없음")


In [ ]:
# ===================== Chronos 모델 로드 =====================
pipeline = BaseChronosPipeline.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype=torch.float32,
)
print(f"Loaded {MODEL_ID} on {DEVICE}")


In [ ]:
# ===================== 변수별 rolling one-step-ahead 예측 (Chronos-2, multivariate) =====================
# 매 시간 위치마다 "그 시점까지의 전체 변수(41개) 과거 데이터"를 한 번에 넣어서 다음 시점의 전체 변수를 예측합니다.
# Chronos-2는 이때 변수들 간의 관계(예: FEED와 온도가 같이 움직이는 패턴)까지 참고해서 각 변수를 예측합니다.
#
# values(컨텍스트용, 글리치 정리됨) vs raw_values(채점용, 원본) 를 분리해서 씁니다:
#  - 예측에 쓰는 과거 168시간 컨텍스트는 정리된 값(values)을 사용 -> 글리치 하나가 그 이후 최대
#    1주일치 예측을 오염시키는 걸 방지
#  - 그 시점 자체를 "이상이다/아니다" 판정할 때 비교하는 실제값은 원본(raw_values)을 그대로 사용
#    -> 글리치 자체도 정상적으로 이상치로 잡히게 함 (숨기지 않음)
#
# 컨텍스트에 NaN(시간 갭)이 섞여 있어도 그대로 넣습니다 (Chronos-2가 결측 마스크로 처리).
# 실제값이 예측 분위수(q_low~q_high) 밖으로 벗어나면 이상치인데, 아래는 이상 "판정" 자체를 제외합니다:
#  - gap   : 타깃 시점 실측값(원본)이 NaN
#  - down  : 설비 정지 시점 (is_down)
#  - warmup: 정지/갭 직후 안정화 구간 (is_warmup, WARMUP_HOURS=0이면 항상 해당 없음)
# 제외된 시점도 예측값(pred_median 등)은 남겨두되 excluded_reason에 사유를 표시하고 is_anomaly는 False로 둡니다.
#
# severity(심각도)는 "예측 구간 폭 대비 벗어난 정도"가 아니라, "그 변수의 정상 운전 구간 표준편차 대비
# 얼마나 벗어났는지"로 계산합니다 (일종의 z-score, variable_scale은 위에서 정지/갭/웜업 제외하고 계산함).

quantile_levels = pipeline.quantiles  # 모델이 고정으로 제공하는 분위수 목록 (예: 0.01~0.99, 21개)


def nearest_quantile_index(target):
    return min(range(len(quantile_levels)), key=lambda k: abs(quantile_levels[k] - target))


low_i = nearest_quantile_index(ANOMALY_QUANTILE_LOW)
mid_i = nearest_quantile_index(0.5)
high_i = nearest_quantile_index(ANOMALY_QUANTILE_HIGH)
print(f"사용하는 분위수: low={quantile_levels[low_i]}, median={quantile_levels[mid_i]}, high={quantile_levels[high_i]}")


def rolling_forecast_multivariate(values: np.ndarray, variable_cols: list, variable_scale: np.ndarray,
                                   is_down: np.ndarray, is_warmup: np.ndarray,
                                   raw_values: np.ndarray = None) -> pd.DataFrame:
    """values: 컨텍스트 구성용(글리치 정리됨), shape (n_timesteps, n_variates), NaN 포함 가능.
    raw_values: 채점(실제값 비교)용 원본 데이터. 지정 안 하면 values와 동일하게 취급.
    variable_scale: shape (n_variates,). is_down/is_warmup: shape (n_timesteps,)."""
    raw_values = raw_values if raw_values is not None else values
    n = values.shape[0]
    values_t = values.T  # (n_variates, n_timesteps) -- 컨텍스트 구성용(정리됨)
    positions = list(range(CONTEXT_LENGTH, n, STRIDE))
    if not positions:
        raise ValueError(
            f"데이터 길이({n})가 CONTEXT_LENGTH({CONTEXT_LENGTH})보다 짧습니다. CONTEXT_LENGTH를 줄여주세요."
        )

    rows = []
    for b in range(0, len(positions), BATCH_WINDOWS):
        batch_pos = positions[b : b + BATCH_WINDOWS]
        batch_inputs = np.stack(
            [values_t[:, i - CONTEXT_LENGTH : i] for i in batch_pos]
        )  # (batch, n_variates, CONTEXT_LENGTH), 정리된 값 기준

        forecasts = pipeline.predict(batch_inputs, prediction_length=PREDICTION_LENGTH)
        # forecasts: list of length batch, each shape (n_variates, n_quantiles, prediction_length)

        for j, i in enumerate(batch_pos):
            fc = forecasts[j].detach().to("cpu").float().numpy()  # (n_variates, n_quantiles, prediction_length)
            for v, col in enumerate(variable_cols):
                actual = float(raw_values[i, v])  # 채점은 항상 원본값 기준
                q_low = float(fc[v, low_i, 0])
                q_mid = float(fc[v, mid_i, 0])
                q_high = float(fc[v, high_i, 0])

                if np.isnan(actual):
                    reason = "gap"
                    error = np.nan
                    severity = np.nan
                    is_anomaly = False
                else:
                    if is_down[i]:
                        reason = "down"
                    elif is_warmup[i]:
                        reason = "warmup"
                    else:
                        reason = None
                    error = actual - q_mid
                    severity = abs(error) / max(float(variable_scale[v]), SEVERITY_EPS)
                    is_anomaly = (actual < q_low or actual > q_high) if reason is None else False

                rows.append(
                    dict(
                        idx=i,
                        variable=col,
                        actual=actual,
                        pred_median=q_mid,
                        pred_low=q_low,
                        pred_high=q_high,
                        error=error,
                        is_anomaly=is_anomaly,
                        severity=severity,
                        excluded_reason=reason,
                    )
                )

    return pd.DataFrame(rows)


In [ ]:
# ===================== 예측 정확도 계산 함수 =====================
# gap/down/warmup 시점은 빼고 계산하고, naive baseline("직전 시간값")은 idx 기준으로 진짜 1시간 전인
# 경우만 비교합니다 (안 그러면 시간갭을 사이에 두고 "직전값"을 비교하는 오류가 생김).
# MAPE는 실제값이 0에 가까우면(예: 0~1 사이를 오가는 변수) 분모가 0에 가까워져서 터무니없이
# 커지는 문제가 있어서, |실제값| <= mape_min_abs인 시점은 MAPE 계산에서만 제외합니다
# (mape_n/mape_skipped로 몇 개를 뺐는지 남겨둠 -- MAPE가 이상하게 크게 나오는 변수는 이 값도 같이 확인).

def compute_accuracy_summary(results_df, mape_min_abs=1e-6):
    rows = []
    for col, sub in results_df.groupby("variable"):
        sub = sub.sort_values("idx").reset_index(drop=True)
        actual = sub["actual"].to_numpy()
        pred = sub["pred_median"].to_numpy()
        scored = sub["excluded_reason"].isna().to_numpy()

        naive_pred = np.roll(actual, 1)
        naive_pred[0] = np.nan
        valid = scored & ~np.isnan(naive_pred)
        n_scored = int(valid.sum())
        if n_scored < 2:
            continue

        actual_v = actual[valid]
        err = actual_v - pred[valid]
        naive_err = actual_v - naive_pred[valid]
        mae = np.mean(np.abs(err))
        naive_mae = np.mean(np.abs(naive_err))
        rmse = np.sqrt(np.mean(err ** 2))

        mape_mask = np.abs(actual_v) > mape_min_abs
        mape_n = int(mape_mask.sum())
        mape_skipped = int((~mape_mask).sum())
        mape = np.mean(np.abs(err[mape_mask] / actual_v[mape_mask])) * 100 if mape_n > 0 else np.nan

        low = sub["pred_low"].to_numpy()[valid]
        high = sub["pred_high"].to_numpy()[valid]
        coverage = ((actual_v >= low) & (actual_v <= high)).mean() * 100

        ss_res = np.sum(err ** 2)
        ss_tot = np.sum((actual_v - actual_v.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

        rows.append(dict(
            variable=col, label=col.replace("\n", " "),
            mae=mae, rmse=rmse, mape=mape, mape_n=mape_n, mape_skipped=mape_skipped,
            naive_mae=naive_mae, mae_vs_naive=mae / naive_mae if naive_mae > 0 else np.nan,
            r2=r2, coverage=coverage, n_scored=n_scored,
        ))
    return pd.DataFrame(rows)


In [ ]:
# ===================== [레거시/비활성] 예전 글리치 raw-clean 비교 =====================
# GLITCH_VARIANTS=[]이므로 실행되지 않습니다. 실제 실행 코드는 바로 다음 review_v2 메인 셀입니다.
# EQUIPMENT_IDS x GLITCH_VARIANTS(정리함/안함)를 전부 순회하면서 각각 독립적으로:
# 로드(reindex) -> 글리치 정리(옵션) -> 정지/갭/웜업 판정 -> Chronos-2 예측 -> 요약/정확도 계산까지
# 수행하고 결과를 딕셔너리에 저장합니다. Run All 한 번으로 두 버전이 다 만들어집니다.
# 모델(pipeline)은 한 번만 로드해서 전부 재사용합니다.
#
# 저장 키(run_key)는 "설비명__glitch_cleaned" 또는 "설비명__glitch_raw" 형태입니다.
#
# 글리치는 "컨텍스트"에서만 정리하고 "채점(실제값 비교)"은 원본 그대로 씁니다 -> 글리치 자체는
# 이상치로 정상적으로 잡히되, 그 이후 최대 1주일치 예측이 오염되는 건 막습니다.
# 어떤 시점의 글리치를 정리했는지는 equipment_results[run_key]["glitch_records"]에 그대로
# 남겨둬서 나중에 "어떤 글리치를 제거했는지" 확인할 수 있게 합니다 (glitch_raw 버전은 당연히 빈 리스트).
#
# 이미 처리한 run_key가 있으면(중간에 멈췄다가 이어서 할 때) 그 결과는 유지하고 나머지만 계산합니다.

try:
    equipment_results  # 이미 있으면(이어서 실행) 기존 결과 유지, 없으면 새로 시작
except NameError:
    equipment_results = {}

for clean_glitches in GLITCH_VARIANTS:
    for equipment_id in EQUIPMENT_IDS:
        run_key = f"{equipment_id}__{'glitch_cleaned' if clean_glitches else 'glitch_raw'}"
        if run_key in equipment_results:
            print(f"\n{run_key}는 이미 처리됨 -> 건너뜀 (다시 하려면 equipment_results에서 지우고 재실행)")
            continue

        print(f"\n{'='*15} {run_key} {'='*15}")

        data, time_index, variable_cols = load_data(
            EXCEL_PATH, SHEET_NAME, HEADER_ROW, DATE_COL, HOUR_COL, EXCLUDE_COLS,
            MIN_OBSERVATION_RATE, equipment_id=equipment_id, id_col=ID_COL,
            predict_exclude_cols=PREDICT_EXCLUDE_COLS,
        )
        if QUICK_TEST_ROWS is not None:
            data = data.tail(QUICK_TEST_ROWS).reset_index(drop=True)
            time_index = time_index.tail(QUICK_TEST_ROWS).reset_index(drop=True)
            print(f"QUICK_TEST_ROWS={QUICK_TEST_ROWS} 적용 -> 최근 {len(data)}시간만 사용")
        print(f"변수 개수: {len(variable_cols)}개 (원본 41개 중 {41-len(variable_cols)}개 자동 제외)")

        data_raw = data.copy()  # 글리치 정리 전 원본 백업 (채점용, 이상 판정에서 그대로 씀)

        # 글리치 정리 (컨텍스트 구성용 data에만 적용, data_raw는 그대로 둠)
        glitch_records = []
        if clean_glitches:
            global_std = data[variable_cols].std()
            for col in variable_cols:
                flags, cleaned = detect_spike_glitches(data[col], global_std[col])
                if flags.any():
                    for i in np.where(flags)[0]:
                        glitch_records.append(dict(
                            variable=col, timestamp=time_index[i],
                            original=data[col].iloc[i], replaced_with=cleaned[i],
                        ))
                    data[col] = cleaned
        print(f"글리치 {len(glitch_records)}건 정리 (clean_glitches={clean_glitches}, "
              f"컨텍스트용에만 반영, 채점은 원본값 사용)")

        # 정지/갭/웜업 판정
        is_gap_row = data.isna().all(axis=1).to_numpy()
        is_down = (data[DOWNTIME_COL] == 0).to_numpy()
        is_down_or_gap = is_down | is_gap_row
        is_warmup = np.zeros(len(data), dtype=bool)
        for i in range(1, len(data)):
            if is_down_or_gap[i - 1] and not is_down_or_gap[i]:
                for w in range(WARMUP_HOURS):
                    if i + w < len(data):
                        is_warmup[i + w] = True
        score_eligible_mask = ~is_down_or_gap & ~is_warmup
        variable_scale = data.loc[score_eligible_mask, variable_cols].std().to_numpy()
        print(f"gap={is_gap_row.sum()}, down={(is_down & ~is_gap_row).sum()}, "
              f"warmup={(is_warmup & ~is_down_or_gap).sum()}, 정상={score_eligible_mask.sum()} / 전체 {len(data)}")

        # Chronos-2 rolling forecast
        values = data[variable_cols].to_numpy(dtype=np.float32)          # 컨텍스트용(정리됨 or 원본)
        raw_values = data_raw[variable_cols].to_numpy(dtype=np.float32)  # 채점용(항상 원본)
        all_results = rolling_forecast_multivariate(
            values, variable_cols, variable_scale, is_down, is_warmup, raw_values=raw_values
        )
        all_results["timestamp"] = [time_index[i] for i in all_results["idx"]]
        results = {col: df.reset_index(drop=True) for col, df in all_results.groupby("variable")}

        scored = all_results[all_results["excluded_reason"].isna()]
        print(f"스코어링 대상 {len(scored)}개 중 이상 {scored['is_anomaly'].sum()}개 "
              f"({scored['is_anomaly'].mean()*100:.2f}%)")

        summary = (
            scored.groupby("variable")
            .agg(n_points=("is_anomaly", "size"), n_anomalies=("is_anomaly", "sum"), max_severity=("severity", "max"))
            .assign(anomaly_rate=lambda d: d["n_anomalies"] / d["n_points"])
            .sort_values("n_anomalies", ascending=False)
        )
        accuracy_summary = compute_accuracy_summary(all_results)

        equipment_results[run_key] = dict(
            equipment_id=equipment_id, clean_data_glitches=clean_glitches,
            data=data, data_raw=data_raw, time_index=time_index, variable_cols=variable_cols,
            is_down=is_down, is_warmup=is_warmup, variable_scale=variable_scale,
            values=values, raw_values=raw_values, glitch_records=glitch_records,
            all_results=all_results, results=results, summary=summary, accuracy_summary=accuracy_summary,
        )

print(f"\n지금까지 처리 완료된 run: {list(equipment_results.keys())}")


In [ ]:
# ===================== GitHub review_v2 방식 메인 파이프라인 =====================
# 1) 고정 물리 기준 위반값만 NaN 처리
# 2) 내수 품종만 유지, 실제 시간 갭은 1시간 grid의 NaN으로 유지
# 3) 품질 변수와 POLYCOM 운전시간은 예측 대상에서 제외
# 4) 운전시간은 past covariate 및 정지 판정에만 사용
# 5) 정리된 값으로 예측과 채점을 모두 수행 (물리 오류값은 성능지표에서 제외)

_prepared_cache = {}
_processed_frames = []
for equipment_id in EQUIPMENT_IDS:
    data_full, time_full, variable_cols, cleaning_audit, prep_stats = load_data(equipment_id)
    _prepared_cache[equipment_id] = (data_full, time_full, variable_cols, cleaning_audit, prep_stats)
    _processed_frames.append(equipment_frame(equipment_id, data_full, time_full, variable_cols))
    print(f"[{equipment_id}] {prep_stats}")

PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
processed_all = pd.concat(_processed_frames, ignore_index=True)
processed_all.to_csv(PREPROCESSED_CSV, index=False, encoding="utf-8-sig")
print(f"전처리 데이터 저장: {PREPROCESSED_CSV} ({len(processed_all):,}행)")

try:
    if any(r.get("protocol") != PROTOCOL_VERSION for r in equipment_results.values()):
        equipment_results = {}
except NameError:
    equipment_results = {}

for equipment_id in EQUIPMENT_IDS:
    run_key = f"{equipment_id}__review_v2"
    if run_key in equipment_results:
        print(f"{run_key}: 이미 처리됨 -> 건너뜀")
        continue

    data, time_index, variable_cols, cleaning_audit, prep_stats = _prepared_cache[equipment_id]
    if QUICK_TEST_ROWS is not None:
        data = data.tail(QUICK_TEST_ROWS).reset_index(drop=True)
        time_index = time_index.tail(QUICK_TEST_ROWS).reset_index(drop=True)

    is_gap_row = data[variable_cols].isna().all(axis=1).to_numpy()
    is_down = data[DOWNTIME_COL].eq(0).fillna(False).to_numpy()
    is_warmup = np.zeros(len(data), dtype=bool)  # review_v2에서는 임의의 재가동 제외시간을 두지 않음
    score_eligible = ~is_gap_row & ~is_down
    variable_scale = data.loc[score_eligible, variable_cols].std().fillna(0.0).to_numpy()

    all_results = rolling_forecast_review_v2(
        pipeline, data, variable_cols, variable_scale, is_down,
        context_length=CONTEXT_LENGTH, prediction_length=PREDICTION_LENGTH,
        stride=STRIDE, batch_windows=BATCH_WINDOWS,
        quantile_low=ANOMALY_QUANTILE_LOW, quantile_high=ANOMALY_QUANTILE_HIGH,
        severity_eps=SEVERITY_EPS,
    )
    all_results["timestamp"] = [time_index[i] for i in all_results["idx"]]
    results = {col: sub.reset_index(drop=True) for col, sub in all_results.groupby("variable")}
    scored = all_results[all_results["excluded_reason"].isna()]
    summary = (
        scored.groupby("variable")
        .agg(n_points=("is_anomaly", "size"), n_anomalies=("is_anomaly", "sum"), max_severity=("severity", "max"))
        .assign(anomaly_rate=lambda frame: frame["n_anomalies"] / frame["n_points"])
        .sort_values("n_anomalies", ascending=False)
    )
    accuracy_summary = compute_accuracy_summary(all_results)
    values = data[variable_cols].to_numpy(dtype=np.float32)

    equipment_results[run_key] = dict(
        protocol=PROTOCOL_VERSION, equipment_id=equipment_id, physical_cleaned=True,
        clean_data_glitches=True, data=data, data_raw=data.copy(), time_index=time_index,
        variable_cols=variable_cols, is_down=is_down, is_warmup=is_warmup,
        variable_scale=variable_scale, values=values, raw_values=values.copy(),
        glitch_records=[], physical_cleaning_count=len(cleaning_audit),
        preprocessing_stats=prep_stats, all_results=all_results, results=results,
        summary=summary, accuracy_summary=accuracy_summary,
    )
    print(f"[{run_key}] targets={len(variable_cols)}, scored={len(scored):,}, "
          f"anomalies={int(scored['is_anomaly'].sum()):,} ({scored['is_anomaly'].mean()*100:.2f}%)")

print(f"완료된 review_v2 run: {list(equipment_results.keys())}")


In [ ]:
# ===================== 변수별 시각화 (실제값 vs 예측값, 점+선) =====================
# matplotlib 스타일. 전체 기간을 한 그래프에 몰아넣지 않고, 7일 단위로 쪼개서
# 한 줄에 1개씩 세로로 쭉 보여줌 — 한 달 단위(약 720시간)는 점이 너무 촘촘해서 더 잘게 쪼갬.
# 7일 구간은 "그 달 안에서" 1~7, 8~14, 15~21, 22~28, 29~말일 식으로 나눕니다 (달력 월 경계를
# 넘기지 않게 매달 안에서 따로 자름 -- 그냥 전체 날짜에 7일씩 이어붙이면 5월 마지막 구간이
# 6월로 넘어가버려서 "5월" 패널에 6월 데이터가 섞이는 문제가 생김).
# y축은 그 구간 데이터만 기준으로 독립적으로 스케일을 잡습니다 (전체 기간 공통 축을 쓰면,
# 다른 구간에 있는 극단치 하나 때문에 축 범위가 확 늘어나서 나머지가 평평하게 눌려 보이는
# 문제가 있었음 -> 실제로 이렇게 나오는 것 발견해서 고침).
# 예측 구간(음영)은 빼고, 실제값(검정 점+선) vs 예측 중앙값(파랑 선)만 겹쳐서 비교합니다.
# 예측값은 원래 움직임이 완만해서 마커 없이 선만 그림 (마커는 스파이크가 중요한 실제값에만 남김).
# x축은 매일(1일 간격)마다 눈금이 보이게 함.
# 이상 탐지된 점만 빨간 테두리로 강조.
#
# 맨 첫 구간은 항상 며칠치만 나올 수 있음(예측을 시작하려면 앞의 CONTEXT_LENGTH(168)시간은
# "과거 참고용"으로만 쓰이고 예측 결과 자체에는 안 나오기 때문 -- 실제 특성이 아니라 표본을 자른
# 방식 때문에 생기는 반쪽짜리 구간이라 분석에 의미가 없음) -> 그 구간 시간의 50% 이상 있어야 표시.
#
# save_path를 주면 화면에 안 띄우고 파일로 저장합니다 (실제 저장은 아래 "전체 run 그래프 이미지
# 한 번에 저장" 셀에서 equipment_results의 모든 run을 순회하며 함, 여기선 함수 정의만).
import math
import os
import matplotlib.dates as mdates


def plot_variable(col, max_periods=None, save_path=None, results_dict=None):
    res = (results_dict if results_dict is not None else results)[col]
    ts = res["timestamp"]
    day, month, year, dim = ts.dt.day, ts.dt.month, ts.dt.year, ts.dt.days_in_month

    chunk = (day - 1) // 7                              # 그 달 안에서 몇 번째 7일 구간인지 (0~4)
    period_id = (year * 12 + (month - 1)) * 5 + chunk    # 달 경계를 넘지 않는 정렬 가능한 키
    chunk_start = chunk * 7 + 1
    chunk_end = np.minimum(chunk_start + 6, dim)         # 마지막 구간은 그 달 말일에서 잘림
    expected_hours = (chunk_end - chunk_start + 1) * 24

    info = pd.DataFrame({
        "period_id": period_id, "year": year, "month": month,
        "start": chunk_start, "end": chunk_end, "expected": expected_hours,
    })
    counts = info.groupby("period_id").size()
    labels = info.drop_duplicates("period_id").set_index("period_id")

    periods = sorted(p for p in counts.index if counts[p] >= labels.loc[p, "expected"] * 0.5)
    if max_periods is not None:
        periods = periods[-max_periods:]

    ncols = 1
    nrows = math.ceil(len(periods) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.2 * nrows), squeeze=False)

    for idx, pid in enumerate(periods):
        ax = axes[idx // ncols][idx % ncols]
        sub = res[period_id == pid]
        y, m, s, e = labels.loc[pid, ["year", "month", "start", "end"]]
        title = f"{y}-{m:02d} ({s}~{e}일)"

        ax.plot(sub["timestamp"], sub["actual"], color="black", linewidth=1,
                 marker="o", markersize=3, label="실제값")
        ax.plot(sub["timestamp"], sub["pred_median"], color="tab:blue", linewidth=1.3, label="예측값")
        anomalies = sub[sub["is_anomaly"]]
        ax.scatter(anomalies["timestamp"], anomalies["actual"], facecolors="none",
                    edgecolors="red", s=55, linewidths=1.4, zorder=5, label="이상 탐지")

        # 이 구간의 데이터만 기준으로 y축 범위 결정 (구간마다 독립적)
        y_range = pd.concat([sub["actual"], sub["pred_median"]]).dropna()
        if len(y_range):
            pad = (y_range.max() - y_range.min()) * 0.1 or 1
            ax.set_ylim(y_range.min() - pad, y_range.max() + pad)

        ax.set_title(title, fontsize=12)
        ax.xaxis.set_major_locator(mdates.DayLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
        ax.tick_params(axis="x", labelrotation=0, labelsize=9)
        ax.tick_params(axis="y", labelsize=9)
        if idx == 0:
            ax.legend(loc="upper left", fontsize=9, framealpha=0.9)

    # 남는 빈 subplot 숨기기
    for idx in range(len(periods), nrows * ncols):
        axes[idx // ncols][idx % ncols].axis("off")

    total_anomalies = res["is_anomaly"].sum()
    fig.suptitle(f"{col} — 실제값 vs Chronos-2 예측 (총 {total_anomalies}개 이상 탐지)", fontsize=14, y=1.0)
    plt.tight_layout(rect=[0, 0, 1, 0.98])

    if save_path:
        fig.savefig(save_path, dpi=120, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()


def sanitize_filename(name: str) -> str:
    """변수명에 '/' 같은 경로 구분자가 섞여 있어서(예: 'MILL C/M출구온도 GAS') 폴더/파일명으로
    바로 못 씀 -> 안전한 문자로 치환."""
    return name.replace("\n", " ").replace("/", "_").strip()


In [ ]:
# ===================== z-score(severity) 시각화 =====================
# 이상 여부(is_anomaly) 자체는 분위수(1~99%)로 판정하지만, "얼마나 벗어났는지"는
# z-score 성격의 severity(변수 자체 표준편차 대비 벗어난 정도)로 따로 보여줍니다.
# 부호(+/-)를 살려서 위/아래 중 어느 쪽으로 벗어났는지도 같이 보이게 함.
# 위 "변수별 시각화"와 동일하게: 7일 단위(달력 월 안에서만 쪼갬)로 세로 1열, 매일 눈금,
# 점 산점도 대신 꺾은선(점+선)으로 표현. 빨간 테두리 = is_anomaly(분위수 기준 실제 이상 판정),
# 빨간 점선 = ±Z_REFERENCE_LINE 참고선(참고용, 판정 기준 아님)
# save_path를 주면 파일로 저장(실제 저장은 "전체 run 그래프 이미지 한 번에 저장" 셀에서
# equipment_results의 모든 run을 순회하며 함, 여기선 함수 정의만).

Z_REFERENCE_LINE = 3.0


def plot_variable_zscore(col, max_periods=None, z_ref=Z_REFERENCE_LINE, save_path=None, results_dict=None):
    res = (results_dict if results_dict is not None else results)[col].copy()
    res["z"] = res["severity"] * np.sign(res["error"])

    ts = res["timestamp"]
    day, month, year, dim = ts.dt.day, ts.dt.month, ts.dt.year, ts.dt.days_in_month
    chunk = (day - 1) // 7
    period_id = (year * 12 + (month - 1)) * 5 + chunk
    chunk_start = chunk * 7 + 1
    chunk_end = np.minimum(chunk_start + 6, dim)
    expected_hours = (chunk_end - chunk_start + 1) * 24

    info = pd.DataFrame({
        "period_id": period_id, "year": year, "month": month,
        "start": chunk_start, "end": chunk_end, "expected": expected_hours,
    })
    counts = info.groupby("period_id").size()
    labels = info.drop_duplicates("period_id").set_index("period_id")

    periods = sorted(p for p in counts.index if counts[p] >= labels.loc[p, "expected"] * 0.5)
    if max_periods is not None:
        periods = periods[-max_periods:]

    ncols = 1
    nrows = math.ceil(len(periods) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 2.8 * nrows), squeeze=False)

    z_max = max(res["z"].abs().max(), z_ref) * 1.1

    for idx, pid in enumerate(periods):
        ax = axes[idx // ncols][idx % ncols]
        sub = res[period_id == pid]
        y, m, s, e = labels.loc[pid, ["year", "month", "start", "end"]]
        title = f"{y}-{m:02d} ({s}~{e}일)"

        ax.plot(sub["timestamp"], sub["z"], color="tab:purple", linewidth=1,
                 marker="o", markersize=3, alpha=0.8)
        anomalies = sub[sub["is_anomaly"]]
        ax.scatter(anomalies["timestamp"], anomalies["z"], facecolors="none",
                    edgecolors="red", s=45, linewidths=1.3, zorder=5, label="이상 탐지")
        ax.axhline(0, color="gray", linewidth=0.7)
        ax.axhline(z_ref, color="red", linestyle="--", linewidth=0.8, alpha=0.6)
        ax.axhline(-z_ref, color="red", linestyle="--", linewidth=0.8, alpha=0.6)

        ax.set_title(title, fontsize=11)
        ax.set_ylim(-z_max, z_max)
        ax.xaxis.set_major_locator(mdates.DayLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
        ax.tick_params(axis="x", labelrotation=0, labelsize=8)
        ax.tick_params(axis="y", labelsize=8)
        if idx == 0:
            ax.legend(loc="upper left", fontsize=8, framealpha=0.9)

    for idx in range(len(periods), nrows * ncols):
        axes[idx // ncols][idx % ncols].axis("off")

    n_anom = res["is_anomaly"].sum()
    fig.suptitle(f"{col} — z-score (severity, 부호 포함) · 총 {n_anom}개 이상", fontsize=13, y=1.0)
    plt.tight_layout(rect=[0, 0, 1, 0.98])

    if save_path:
        fig.savefig(save_path, dpi=120, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()


In [ ]:
# ===================== 설비별 review_v2 결과 비교 요약 =====================
# R² / coverage / MAE 기준으로 비교 (MSE/RMSE는 이상치 하나에도 크게 튀어서 제외)
compare_rows = []
for run_key, r in equipment_results.items():
    scored = r["all_results"][r["all_results"]["excluded_reason"].isna()]
    acc = compute_accuracy_summary(r["all_results"])
    compare_rows.append(dict(
        run=run_key,
        설비=r["equipment_id"],
        물리기준정리=r.get("physical_cleaned", False),
        변수수=len(r["variable_cols"]),
        스코어링대상=len(scored),
        이상탐지수=int(scored["is_anomaly"].sum()),
        이상비율=scored["is_anomaly"].mean() * 100,
        물리기준마스킹건수=r.get("physical_cleaning_count", 0),
        평균_R2=acc["r2"].mean(),
        중앙값_R2=acc["r2"].median(),
        평균_coverage=acc["coverage"].mean(),
        평균_MAE=acc["mae"].mean(),
    ))
equipment_comparison = pd.DataFrame(compare_rows).set_index("run")
print(equipment_comparison.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].plot(equipment_comparison.index, equipment_comparison["이상비율"], marker="o", color="tab:blue", linewidth=1.5)
axes[0].set_title("run별 이상 비율 (%)")
axes[1].plot(equipment_comparison.index, equipment_comparison["평균_R2"], marker="o", color="tab:green", linewidth=1.5)
axes[1].set_title("run별 평균 R²")
axes[2].plot(equipment_comparison.index, equipment_comparison["평균_coverage"], marker="o", color="tab:purple", linewidth=1.5)
axes[2].axhline((ANOMALY_QUANTILE_HIGH - ANOMALY_QUANTILE_LOW) * 100, color="black", linestyle="--", linewidth=1)
axes[2].set_title("run별 평균 coverage (%)")
for ax in axes:
    ax.tick_params(axis="x", labelrotation=30, labelsize=8)
plt.tight_layout()
plt.show()

equipment_comparison


In [ ]:
# ===================== 자기상관(ACF) vs R²(글리치 정리 전/후) 비교 =====================
# PPT용 핵심 그래프: "자기상관이 낮으면 R²도 낮다 -> 그래서 시계열 예측이 어렵다"를
# 증명하기 위해, 변수별 ACF(lag-1) 옆에 R²를 글리치 정리 전(raw)/후(cleaned) 두 버전 다 같이 놓고 봅니다.
# 글리치를 정리해도(컨텍스트만 깨끗해짐, 채점은 항상 원본과 비교) ACF 낮은 변수는 여전히 R²가
# 낮게 나온다는 걸 보여주면 -> "이건 데이터 오류 때문이 아니라 원래 예측이 어려운 변수다"라는
# 결론이 뒷받침됩니다.
# equipment_results에 "{설비}__glitch_raw"와 "{설비}__glitch_cleaned"가 둘 다 있어야 그릴 수 있음
# (아직 둘 다 안 끝난 설비는 건너뜀).

def _acf1_map(data_raw, variable_cols):
    def acf1(x):
        a, b = x[1:], x[:-1]
        mask = ~np.isnan(a) & ~np.isnan(b)
        if mask.sum() < 10:
            return np.nan
        return np.corrcoef(a[mask], b[mask])[0, 1]
    return {col: acf1(data_raw[col].to_numpy(dtype=np.float64)) for col in variable_cols}


def plot_acf_vs_r2_comparison(equipment_id, save_path=None):
    key_raw, key_clean = f"{equipment_id}__glitch_raw", f"{equipment_id}__glitch_cleaned"
    r_raw, r_clean = equipment_results[key_raw], equipment_results[key_clean]

    acc_raw = compute_accuracy_summary(r_raw["all_results"]).set_index("variable")["r2"]
    acc_clean = compute_accuracy_summary(r_clean["all_results"]).set_index("variable")["r2"]
    acf1_map = _acf1_map(r_raw["data_raw"], r_raw["variable_cols"])

    rows = [
        dict(variable=col, label=col.replace("\n", " "), acf1=acf1_map.get(col, np.nan),
             r2_raw=acc_raw.get(col, np.nan), r2_cleaned=acc_clean.get(col, np.nan))
        for col in r_raw["variable_cols"]
    ]
    comp = pd.DataFrame(rows).dropna(subset=["acf1"]).sort_values("acf1", ascending=False).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(max(10, len(comp) * 0.35), 5))
    x = range(len(comp))
    ax.plot(x, comp["acf1"], marker="o", markersize=4, color="dimgray", linewidth=1.6, label="자기상관 ACF(lag-1)")
    ax.plot(x, comp["r2_raw"], marker="o", markersize=4, color="tab:red", linewidth=1.2,
            linestyle="--", label="R² (글리치 미정리)")
    ax.plot(x, comp["r2_cleaned"], marker="o", markersize=4, color="tab:blue", linewidth=1.2,
            label="R² (글리치 정리)")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axhline(0.5, color="gray", linewidth=0.8, linestyle=":")
    ax.set_xticks(list(x))
    ax.set_xticklabels(comp["label"], rotation=90, fontsize=7)
    ax.set_title(f"[{equipment_id}] 자기상관(ACF) vs R²(글리치 정리 전/후) -- 변수는 ACF 높은 순 정렬")
    ax.legend(loc="upper right", fontsize=9)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=120, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()

    valid = comp.dropna(subset=["r2_cleaned"])
    rho_clean = np.corrcoef(valid["acf1"].rank(), valid["r2_cleaned"].rank())[0, 1]
    rho_raw = np.corrcoef(valid["acf1"].rank(), valid["r2_raw"].rank())[0, 1]
    print(f"[{equipment_id}] Spearman(ACF, R²_raw)={rho_raw:.3f}, Spearman(ACF, R²_cleaned)={rho_clean:.3f}")
    return comp


os.makedirs("image", exist_ok=True)
for _eid in EQUIPMENT_IDS:
    _key_raw, _key_clean = f"{_eid}__glitch_raw", f"{_eid}__glitch_cleaned"
    if _key_raw in equipment_results and _key_clean in equipment_results:
        plot_acf_vs_r2_comparison(_eid, save_path=os.path.join("image", f"{_eid}_acf_vs_r2.png"))
        print(f"저장 완료: image/{_eid}_acf_vs_r2.png")
    else:
        print(f"{_eid}: 아직 준비 안 됨 (raw={_key_raw in equipment_results}, cleaned={_key_clean in equipment_results})")


In [ ]:
# ===================== 전체 run(설비 x 글리치처리 여부) CSV 한 번에 저장 =====================
# CURRENT_RUN을 매번 손으로 바꿔가며 "결과 저장"/"예측값 wide 저장" 셀을 반복하는 대신,
# equipment_results에 있는 모든 run을 한 번에 순회하면서 long/wide 두 형식 다 저장합니다.
cols_to_save = ["variable", "timestamp", "actual", "pred_median", "pred_low", "pred_high",
                "error", "severity", "is_anomaly", "excluded_reason"]

for _run_key, _r in equipment_results.items():
    _all_results = _r["all_results"]

    # long format
    _export_df = _all_results[cols_to_save].copy()
    _export_df["variable"] = _export_df["variable"].str.replace("\n", " ")
    _long_path = f"chronos_anomaly_results_{_run_key}.csv"
    _export_df.to_csv(_long_path, index=False, encoding="utf-8-sig")

    # wide format (원본 엑셀 형식, 예측 중앙값 기준)
    _wide = _all_results.copy()
    _wide["variable"] = _wide["variable"].str.replace("\n", " ")
    _wide = _wide.pivot(index="timestamp", columns="variable", values="pred_median").reset_index()
    _wide.insert(0, "ID", _r["equipment_id"])
    _wide.insert(1, "근무일자", _wide["timestamp"].dt.strftime("%Y-%m-%d"))
    _wide.insert(2, "근무시간", _wide["timestamp"].dt.hour)
    _wide = _wide.drop(columns=["timestamp"])
    _wide_path = f"chronos_predicted_원본형식_{_run_key}.csv"
    _wide.to_csv(_wide_path, index=False, encoding="utf-8-sig")

    print(f"저장 완료: {_long_path} ({len(_export_df)}행), {_wide_path} ({_wide.shape[0]}행 x {_wide.shape[1]}컬럼)")

print(f"\n총 {len(equipment_results)}개 run 저장 완료: {list(equipment_results.keys())}")


In [ ]:
# ===================== 전체 run(설비 x 글리치처리 여부) 그래프 이미지 한 번에 저장 =====================
# CURRENT_RUN을 손으로 바꿔가며 "변수별 시각화"/"z-score 시각화" 셀을 6번 반복하는 대신,
# equipment_results에 있는 모든 run을 자동으로 순회하면서 두 종류(실제vs예측, z-score) 그래프를
# 전부 image/{glitch_clean|glitch}/{설비}/{변수}/ 아래에 저장합니다.
# 전체 기간(5.8년) x 36~37개 변수 x 6개 run x 2종류라서 시간이 꽤 걸립니다 (변수별 시각화 셀
# 하나 도는 시간의 대략 6배 정도로 예상).

for _run_key, _r in equipment_results.items():
    _run_variable_cols = _r["variable_cols"]
    _run_results = _r["results"]
    _run_image_dir = os.path.join("image", "review_v2_physical_cleaned", _r["equipment_id"])

    for col in _run_variable_cols:
        _safe_name = sanitize_filename(col)
        _var_dir = os.path.join(_run_image_dir, _safe_name)
        os.makedirs(_var_dir, exist_ok=True)
        plot_variable(col, save_path=os.path.join(_var_dir, f"{_safe_name}.png"),
                      results_dict=_run_results)
        plot_variable_zscore(col, save_path=os.path.join(_var_dir, f"{_safe_name}_zscore.png"),
                             results_dict=_run_results)

    print(f"[{_run_key}] 변수 {len(_run_variable_cols)}개 그래프 저장 완료 -> {_run_image_dir}/")

print(f"\n전체 {len(equipment_results)}개 run 이미지 저장 완료")


In [ ]:
# ===================== [레거시/비활성] 예전 glitch CSV 복구 =====================
# 커널이 재시작돼서 equipment_results가 사라졌는데, 무거운 Chronos rolling forecast(설비당
# 1.5~2시간)를 처음부터 다시 돌리고 싶지 않을 때 씁니다. 이미 저장된
# chronos_anomaly_results_{run}.csv에 zero-shot 예측 결과가 다 들어있으니, 그건 그대로 불러오고
# 데이터 로드 + 글리치 정리(둘 다 수십 초 내로 끝나는 가벼운 작업)만 다시 해서
# equipment_results[run_key]를 원래(828d63e9가 만드는 것)와 완전히 동일한 형태로 복원합니다.
# 아래 "설비별 파이프라인 실행" 셀 대신 이 셀만 쓰면 해당 run 하나를 빠르게 되살릴 수 있습니다.

def rebuild_equipment_result(equipment_id, clean_glitches):
    run_key = f"{equipment_id}__{'glitch_cleaned' if clean_glitches else 'glitch_raw'}"
    data, time_index, variable_cols = load_data(
        EXCEL_PATH, SHEET_NAME, HEADER_ROW, DATE_COL, HOUR_COL, EXCLUDE_COLS,
        MIN_OBSERVATION_RATE, equipment_id=equipment_id, id_col=ID_COL,
    )
    data_raw = data.copy()

    glitch_records = []
    if clean_glitches:
        global_std = data[variable_cols].std()
        for col in variable_cols:
            flags, cleaned = detect_spike_glitches(data[col], global_std[col])
            if flags.any():
                for i in np.where(flags)[0]:
                    glitch_records.append(dict(
                        variable=col, timestamp=time_index[i],
                        original=data[col].iloc[i], replaced_with=cleaned[i],
                    ))
                data[col] = cleaned

    is_gap_row = data.isna().all(axis=1).to_numpy()
    is_down = (data[DOWNTIME_COL] == 0).to_numpy()
    is_down_or_gap = is_down | is_gap_row
    is_warmup = np.zeros(len(data), dtype=bool)
    for i in range(1, len(data)):
        if is_down_or_gap[i - 1] and not is_down_or_gap[i]:
            for w in range(WARMUP_HOURS):
                if i + w < len(data):
                    is_warmup[i + w] = True
    score_eligible_mask = ~is_down_or_gap & ~is_warmup
    variable_scale = data.loc[score_eligible_mask, variable_cols].std().to_numpy()

    values = data[variable_cols].to_numpy(dtype=np.float32)
    raw_values = data_raw[variable_cols].to_numpy(dtype=np.float32)

    # Chronos 재추론 없이, 저장된 CSV에서 zero-shot 예측 결과를 그대로 불러옴
    all_results = pd.read_csv(f"chronos_anomaly_results_{run_key}.csv")
    all_results["timestamp"] = pd.to_datetime(all_results["timestamp"])
    idx_map = pd.Series(np.arange(len(time_index)), index=time_index.to_numpy())
    all_results["idx"] = idx_map.reindex(all_results["timestamp"].to_numpy()).to_numpy()
    results = {col: df.reset_index(drop=True) for col, df in all_results.groupby("variable")}

    scored = all_results[all_results["excluded_reason"].isna()]
    summary = (
        scored.groupby("variable")
        .agg(n_points=("is_anomaly", "size"), n_anomalies=("is_anomaly", "sum"), max_severity=("severity", "max"))
        .assign(anomaly_rate=lambda d: d["n_anomalies"] / d["n_points"])
        .sort_values("n_anomalies", ascending=False)
    )
    accuracy_summary = compute_accuracy_summary(all_results)

    return run_key, dict(
        equipment_id=equipment_id, clean_data_glitches=clean_glitches,
        data=data, data_raw=data_raw, time_index=time_index, variable_cols=variable_cols,
        is_down=is_down, is_warmup=is_warmup, variable_scale=variable_scale,
        values=values, raw_values=raw_values, glitch_records=glitch_records,
        all_results=all_results, results=results, summary=summary, accuracy_summary=accuracy_summary,
    )


try:
    equipment_results
except NameError:
    equipment_results = {}

# 이 셀은 예전 glitch CSV 복구용 레거시 코드이므로 review_v2에서는 자동 실행하지 않는다.
# 새 결과는 위 메인 파이프라인 또는 chronos_anomaly_results_{설비}__review_v2.csv를 사용한다.
for _eid, _clean in []:
    _run_key, _result = rebuild_equipment_result(_eid, _clean)
    equipment_results[_run_key] = _result
    print(f"복구 완료: {_run_key} (Chronos 재추론 없이 CSV + 가벼운 전처리만으로 재구성함)")

print(f"\n현재 equipment_results 키: {list(equipment_results.keys())}")


In [ ]:
# ===================== 상세 분석할 review_v2 run 선택 =====================
# CURRENT_RUN의 설비명만 바꾸면 2CM/3CM/4CM 결과를 전환할 수 있습니다.
# accuracy_summary는 캐시된 값을 안 쓰고 매번 새로 계산합니다 (Chronos 재추론 없이 순수 집계라
# 빠르고, compute_accuracy_summary 함수를 수정했을 때 캐시가 옛날 버전이라 안 바뀌는 문제를 방지).
CURRENT_RUN = "2CM__review_v2"

_r = equipment_results[CURRENT_RUN]
CURRENT_EQUIPMENT = _r["equipment_id"]  # 원래 설비명만 (파일명/ID 컬럼용)
data, data_raw, variable_cols = _r["data"], _r["data_raw"], _r["variable_cols"]
variable_scale = _r["variable_scale"]
all_results, results = _r["all_results"], _r["results"]
summary = _r["summary"]
accuracy_summary = compute_accuracy_summary(all_results)

print(f"현재 상세 보기: {CURRENT_RUN} (설비={CURRENT_EQUIPMENT}, "
      f"물리 기준 NaN 처리 {_r['physical_cleaning_count']}건)")
summary


In [ ]:
# ===================== 예측 정확도 시각화 =====================
# R² / coverage / MAE / MAPE 네 가지를 변수별로 꺾은선 그래프로 봅니다 (변수는 R² 높은 순으로 정렬).
# - R²: 변수마다 스케일이 달라서 변수 간 비교에 제일 적합 (0.5 이상=잘 맞음, 음수=평균 찍는 것보다 못함).
#   자기상관(ACF)과의 관계를 보여주는 핵심 지표라서 메인으로 씀.
# - coverage: 예측 구간(1~99%)에 실제값이 들어온 비율 -- 이상탐지 판정 기준 자체가 잘 맞는지 확인
# - MAE: 실제 단위로 "평소 얼마나 차이나는지" 감 잡는 용도 (이상치 몇 개에 크게 안 흔들림)
# - MAPE: 상대오차(%) -- 변수 간 스케일 차이를 좀 더 보정해서 보고 싶을 때 참고용 (multi-horizon
#   실험 쪽과 비교하려고 추가함). 단, 실제값이 0 근처인 변수는 분모가 작아져서 값이 터무니없이
#   커질 수 있음 (accuracy_summary의 mape_skipped로 몇 개 뺐는지 확인 가능).
# RMSE는 뺐음 -- 극단치 하나에 완전히 흔들려서(조분량 사례: 3천만짜리 값 하나로 RMSE가 159,422까지
# 튐) 변수 간 비교용 메인 지표로는 부적합하다고 판단함 (accuracy_summary["rmse"]로 따로 확인 가능).
#
# RMSE가 전체 중앙값 대비 압도적으로 큰 변수는 (다른 그래프에서도) 따로 뺌 -- 그 변수 하나가
# 축을 다 눌러버려서 나머지 변수들끼리 비교가 안 되는 문제가 있었음. 이런 변수는 메인 그래프에서
# 빼고 별도 표로 보여줌 (accuracy_summary는 위 "상세 분석할 설비 선택" 셀에서 이미 계산되어 있음).

def plot_accuracy_summary(acc=None, outlier_rmse_multiplier=20):
    # acc=None이면 호출 시점의 최신 accuracy_summary를 씀 (함수 정의 시점 값으로
    # 고정되면 계산 함수를 고쳐도 '설비 선택' 셀을 다시 안 돌리면 옛날 값이 남는 문제가 있었음)
    acc = accuracy_summary if acc is None else acc
    target_coverage = (ANOMALY_QUANTILE_HIGH - ANOMALY_QUANTILE_LOW) * 100
    acc = acc.dropna(subset=["r2"])

    median_rmse = acc["rmse"].median()
    is_extreme = acc["rmse"] > median_rmse * outlier_rmse_multiplier
    acc_extreme = acc[is_extreme]
    acc_main = acc[~is_extreme]

    acc_sorted = acc_main.sort_values("r2", ascending=False).reset_index(drop=True)
    x = range(len(acc_sorted))

    fig, axes = plt.subplots(4, 1, figsize=(max(10, len(acc_sorted) * 0.35), 14), sharex=True)

    ax = axes[0]
    ax.plot(x, acc_sorted["r2"], marker="o", markersize=4, color="tab:green", linewidth=1.2)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axhline(0.5, color="gray", linewidth=0.8, linestyle="--")
    ax.set_title("R² (설명력) -- 0.5 이상=잘 맞음, 음수=평균 찍는 것보다 못함")
    ax.tick_params(axis="y", labelsize=8)

    ax = axes[1]
    ax.plot(x, acc_sorted["coverage"], marker="o", markersize=4, color="tab:purple", linewidth=1.2)
    ax.axhline(target_coverage, color="black", linewidth=1, linestyle="--")
    ax.set_title(f"예측구간 커버리지 % (점선={target_coverage:.0f}% 이상적)")
    ax.tick_params(axis="y", labelsize=8)

    ax = axes[2]
    ax.plot(x, acc_sorted["mae"], marker="o", markersize=4, color="tab:blue", linewidth=1.2)
    ax.set_title("MAE (실제 단위)")
    ax.tick_params(axis="y", labelsize=8)

    ax = axes[3]
    ax.plot(x, acc_sorted["mape"], marker="o", markersize=4, color="tab:orange", linewidth=1.2)
    ax.set_title("MAPE (%)")
    ax.tick_params(axis="y", labelsize=8)

    # sharex=True를 쓰면 매트플롯립이 기본적으로 맨 아래 그래프에만 x축 라벨(변수명)을 남기고
    # 위쪽은 다 지워버림 -> 매번 맨 아래까지 스크롤해서 봐야 하는 게 불편해서, 모든 패널에
    # 변수명이 다 보이게 강제로 다시 켜줌.
    for ax in axes:
        ax.set_xticks(list(x))
        ax.set_xticklabels(acc_sorted["label"], rotation=90, fontsize=7)
        ax.tick_params(axis="x", labelbottom=True)

    plt.tight_layout()
    plt.show()

    print(f"전체 평균(아래 극단치 제외, {len(acc_main)}개 기준): "
          f"R²={acc_main['r2'].mean():.3f}(중앙값 {acc_main['r2'].median():.3f}), "
          f"coverage={acc_main['coverage'].mean():.2f}%(목표 {target_coverage:.0f}%), "
          f"MAE={acc_main['mae'].mean():.3f}, MAPE={acc_main['mape'].mean():.2f}%")
    print(f"R² >= 0.5인 변수: {(acc_main['r2'] >= 0.5).sum()}/{len(acc_main)}")

    if len(acc_extreme):
        print(f"\n[그래프에서 제외한 극단치 변수 {len(acc_extreme)}개] "
              f"RMSE가 전체 중앙값({median_rmse:.2f})의 {outlier_rmse_multiplier}배 초과 "
              f"-- 데이터 자체에 물리적으로 불가능한 극단값이 섞여있을 가능성이 높음, 별도 확인 필요:")
        display(acc_extreme[["label", "r2", "coverage", "mae", "mape", "n_scored"]])

    print("\n변수별 상세 (R² 높은 순, 극단치 제외):")
    display(acc_sorted[["label", "r2", "coverage", "mae", "mape", "n_scored"]])


plot_accuracy_summary()


In [ ]:
# ===================== 변수별 자기상관(ACF) 분석 =====================
# 각 변수가 "자기 자신의 과거값과 얼마나 관련 있는지"(자기상관, autocorrelation)를 봅니다.
# ACF가 낮으면(0 근처) 그 변수는 과거값만으로는 예측하기 어렵다는 뜻 -- Chronos R²가 낮게 나온
# 변수와 실제로 자기상관도 낮은지 서로 대조해볼 수 있음.
#
# 시간 갭(reindex로 NaN 처리된 부분)이 있어서 pandas의 `.dropna().autocorr(lag=k)`는 쓰면 안 됨
# (dropna를 먼저 하면 배열이 압축되면서 "lag k"가 실제 시간 간격과 안 맞게 됨 -- 실제로 확인된 버그).
# 대신 압축하지 않고 "실제 시간 위치" 기준으로 두 값이 모두 있는 쌍만 골라서(pairwise deletion)
# 상관계수를 계산합니다.

def compute_acf(x: np.ndarray, max_lag: int) -> np.ndarray:
    """x: 1시간 grid 기준 값 배열(NaN 포함 가능). 반환: lag 0~max_lag의 ACF 배열."""
    acf = np.full(max_lag + 1, np.nan)
    for k in range(max_lag + 1):
        if k == 0:
            acf[k] = 1.0
            continue
        a, b = x[k:], x[:-k]
        mask = ~np.isnan(a) & ~np.isnan(b)
        if mask.sum() < 10:
            continue
        acf[k] = np.corrcoef(a[mask], b[mask])[0, 1]
    return acf


ACF_MAX_LAG = 336  # 2주 (24 * 14)

_acf_by_var = {col: compute_acf(data[col].to_numpy(dtype=np.float64), ACF_MAX_LAG) for col in variable_cols}
acf_lag1 = pd.Series({col: v[1] for col, v in _acf_by_var.items()}, name="acf_lag1")
print(f"[{CURRENT_EQUIPMENT}] 변수 {len(acf_lag1)}개 자기상관(lag 0~{ACF_MAX_LAG}h) 계산 완료")


def plot_acf_overview(acf_series=None):
    acf_series = acf_lag1 if acf_series is None else acf_series
    s = acf_series.dropna().sort_values(ascending=False)
    labels = [c.replace("\n", " ") for c in s.index]

    fig, ax = plt.subplots(figsize=(max(10, len(s) * 0.35), 4))
    ax.plot(range(len(s)), s.values, marker="o", markersize=4, color="teal", linewidth=1.2)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axhline(0.5, color="gray", linewidth=0.8, linestyle="--")
    ax.set_xticks(range(len(s)))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_title(f"[{CURRENT_EQUIPMENT}] 변수별 lag-1 자기상관(ACF) "
                 f"-- 높을수록 바로 직전 값 하나로도 다음 값 예측이 쉬움")
    plt.tight_layout()
    plt.show()


def plot_acf_detail(col, max_lag=ACF_MAX_LAG):
    label = col.replace("\n", " ")
    acf = _acf_by_var[col]
    n = data[col].notna().sum()
    ci = 1.96 / np.sqrt(n)

    fig, ax = plt.subplots(figsize=(12, 3.5))
    ax.bar(range(max_lag + 1), acf, width=0.8, color="tab:blue")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axhline(ci, color="gray", linestyle="--", linewidth=0.8)
    ax.axhline(-ci, color="gray", linestyle="--", linewidth=0.8)
    for k in (24, 168, 336):
        if k <= max_lag:
            ax.axvline(k, color="lightgray", linewidth=1, zorder=0)
    ax.set_title(f"{label} -- ACF(lag 0~{max_lag}h), 점선=95% 신뢰구간(±1.96/sqrt(n)), "
                 f"회색 세로선=24h/168h/336h")
    ax.set_xlabel("lag (시간)")
    plt.tight_layout()
    plt.show()


plot_acf_overview()

# R² 높은/낮은 변수 몇 개를 골라 자기상관 구조를 자세히 대조 (accuracy_summary는 위에서 이미 계산됨)
_acf_top = accuracy_summary.sort_values("r2", ascending=False)["variable"].head(2).tolist()
_acf_bottom = accuracy_summary.sort_values("r2", ascending=True)["variable"].head(2).tolist()
print("\nR² 높은 변수(예측 잘 됨) vs 낮은 변수(예측 잘 안 됨)의 자기상관 구조 비교:")
for _col in _acf_top + _acf_bottom:
    plot_acf_detail(_col)


# [보류] 다중 Horizon 실험

현재 공식 실행은 요청대로 `PREDICTION_LENGTH=1`만 사용합니다. 아래의 예전 1~3시간 실험 코드는
참고용으로 남기되 자동 실행 대상 설비를 비워 두었습니다.

- 글리치 정리(cleaned) 버전만 사용 (raw와의 비교는 이미 다른 실험에서 검증 완료)
- 2CM/3CM/4CM 전체 기간
- 이 섹션은 **독립적으로 동작**합니다 -- 아래 셀 실행 전에 "import" / "CONFIG" / "데이터 로드 함수"
  / "글리치 탐지 함수" / "Chronos 모델 로드" 셀만 실행되어 있으면 되고, 기존 메인 파이프라인
  (`equipment_results`)이나 그 이후 셀들은 몰라도 됩니다.
- 지난번 크래시 경험 때문에 여기서는 거대한 시계열 이미지는 안 만들고, horizon별 정확도
  수치/그래프만 뽑습니다.


In [ ]:
# ===================== [실험] 설정 + horizon별 rolling forecast 함수 =====================
MULTISTEP_PREDICTION_LENGTH = 3       # 1~3시간 뒤를 한 번에 예측
MULTISTEP_EQUIPMENT_IDS = []  # 공식 파이프라인은 prediction_length=1이므로 자동 실행하지 않음

# 기존 pipeline.quantiles 기반 분위수 인덱스 재사용 (이미 위에서 low_i/mid_i/high_i가 정의돼
# 있으면 그대로 쓰고, 없으면(이 섹션만 독립적으로 실행한 경우) 여기서 새로 계산)
quantile_levels = pipeline.quantiles


def _nearest_quantile_index(target):
    return min(range(len(quantile_levels)), key=lambda k: abs(quantile_levels[k] - target))


low_i = _nearest_quantile_index(ANOMALY_QUANTILE_LOW)
mid_i = _nearest_quantile_index(0.5)
high_i = _nearest_quantile_index(ANOMALY_QUANTILE_HIGH)


def rolling_forecast_multihorizon(values, variable_cols, variable_scale, is_down, raw_values, prediction_length):
    """values: 컨텍스트용(글리치 정리됨). raw_values: 채점용(원본).
    한 컨텍스트당 prediction_length개의 미래 시점을 한 번에 예측하고,
    horizon(1, 2, ..., prediction_length)별로 결과를 따로 남깁니다."""
    n = values.shape[0]
    values_t = values.T
    positions = list(range(CONTEXT_LENGTH, n - prediction_length + 1, STRIDE))
    if not positions:
        raise ValueError(f"데이터 길이({n})가 너무 짧습니다 (CONTEXT_LENGTH+prediction_length 필요).")

    rows = []
    for b in range(0, len(positions), BATCH_WINDOWS):
        batch_pos = positions[b: b + BATCH_WINDOWS]
        batch_inputs = np.stack([values_t[:, i - CONTEXT_LENGTH:i] for i in batch_pos])
        forecasts = pipeline.predict(batch_inputs, prediction_length=prediction_length)

        for j, i in enumerate(batch_pos):
            fc = forecasts[j].detach().to("cpu").float().numpy()  # (n_variates, n_quantiles, prediction_length)
            for h in range(1, prediction_length + 1):
                target_idx = i + h - 1
                for v, col in enumerate(variable_cols):
                    actual = float(raw_values[target_idx, v])
                    q_low = float(fc[v, low_i, h - 1])
                    q_mid = float(fc[v, mid_i, h - 1])
                    q_high = float(fc[v, high_i, h - 1])

                    if np.isnan(actual):
                        reason, error, severity, is_anomaly = "gap", np.nan, np.nan, False
                    else:
                        reason = "down" if is_down[target_idx] else None
                        error = actual - q_mid
                        severity = abs(error) / max(float(variable_scale[v]), SEVERITY_EPS)
                        is_anomaly = (actual < q_low or actual > q_high) if reason is None else False

                    rows.append(dict(
                        idx=target_idx, horizon=h, variable=col, actual=actual,
                        pred_median=q_mid, pred_low=q_low, pred_high=q_high,
                        error=error, is_anomaly=is_anomaly, severity=severity, excluded_reason=reason,
                    ))
    return pd.DataFrame(rows)


In [ ]:
# ===================== [실험] 설비별 horizon 예측 실행 (2CM/3CM/4CM, 글리치 정리, 전체 기간) =====================
multistep_results = {}  # 이전 커널의 3시간 실험 결과도 공식 1시간 실행에 섞지 않음

for equipment_id in MULTISTEP_EQUIPMENT_IDS:
    if equipment_id in multistep_results:
        print(f"\n{equipment_id}는 이미 처리됨 -> 건너뜀 (다시 하려면 multistep_results에서 지우고 재실행)")
        continue

    print(f"\n{'='*15} {equipment_id} (horizon 1~{MULTISTEP_PREDICTION_LENGTH}) {'='*15}")

    data, time_index, variable_cols = load_data(
        EXCEL_PATH, SHEET_NAME, HEADER_ROW, DATE_COL, HOUR_COL, EXCLUDE_COLS,
        MIN_OBSERVATION_RATE, equipment_id=equipment_id, id_col=ID_COL,
        predict_exclude_cols=PREDICT_EXCLUDE_COLS,
    )
    print(f"변수 개수: {len(variable_cols)}개")

    data_raw = data.copy()
    global_std = data[variable_cols].std()
    for col in variable_cols:
        flags, cleaned = detect_spike_glitches(data[col], global_std[col])
        if flags.any():
            data[col] = cleaned
    print("글리치 정리 완료 (컨텍스트용에만 반영)")

    is_gap_row = data.isna().all(axis=1).to_numpy()
    is_down = (data[DOWNTIME_COL] == 0).to_numpy() | is_gap_row
    variable_scale = data.loc[~is_down, variable_cols].std().to_numpy()

    values = data[variable_cols].to_numpy(dtype=np.float32)
    raw_values = data_raw[variable_cols].to_numpy(dtype=np.float32)

    all_results_ms = rolling_forecast_multihorizon(
        values, variable_cols, variable_scale, is_down, raw_values, MULTISTEP_PREDICTION_LENGTH
    )
    all_results_ms["timestamp"] = [time_index[i] for i in all_results_ms["idx"]]

    for h in range(1, MULTISTEP_PREDICTION_LENGTH + 1):
        scored_h = all_results_ms[(all_results_ms["horizon"] == h) & all_results_ms["excluded_reason"].isna()]
        print(f"  horizon={h}h: 스코어링 대상 {len(scored_h)}개 중 이상 {scored_h['is_anomaly'].sum()}개 "
              f"({scored_h['is_anomaly'].mean()*100:.2f}%)")

    multistep_results[equipment_id] = dict(
        variable_cols=variable_cols, all_results=all_results_ms,
    )

print(f"\n처리 완료된 설비: {list(multistep_results.keys())}")


In [ ]:
# ===================== [실험] horizon별 정확도 계산 + 비교 그래프 =====================
def compute_accuracy_by_horizon(all_results_ms, mape_min_abs=1e-6):
    rows = []
    for (col, h), sub in all_results_ms.groupby(["variable", "horizon"]):
        sub = sub.sort_values("idx").reset_index(drop=True)
        actual = sub["actual"].to_numpy()
        pred = sub["pred_median"].to_numpy()
        scored = sub["excluded_reason"].isna().to_numpy()
        if scored.sum() < 2:
            continue
        actual_v, pred_v = actual[scored], pred[scored]
        err = actual_v - pred_v
        mae = np.mean(np.abs(err))
        rmse = np.sqrt(np.mean(err ** 2))

        mape_mask = np.abs(actual_v) > mape_min_abs
        mape = np.mean(np.abs(err[mape_mask] / actual_v[mape_mask])) * 100 if mape_mask.sum() > 0 else np.nan

        ss_res = np.sum(err ** 2)
        ss_tot = np.sum((actual_v - actual_v.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
        low, high = sub["pred_low"].to_numpy()[scored], sub["pred_high"].to_numpy()[scored]
        coverage = ((actual_v >= low) & (actual_v <= high)).mean() * 100
        rows.append(dict(variable=col, label=col.replace("\n", " "), horizon=h,
                          r2=r2, mae=mae, rmse=rmse, mape=mape, coverage=coverage, n_scored=int(scored.sum())))
    return pd.DataFrame(rows)


multistep_accuracy = {}
for eid, r in multistep_results.items():
    acc_h = compute_accuracy_by_horizon(r["all_results"])
    multistep_accuracy[eid] = acc_h

    summary_by_h = acc_h.groupby("horizon")[["r2", "mae", "rmse", "mape", "coverage"]].mean()
    print(f"\n[{eid}] horizon별 평균 (전체 변수 평균, RMSE/MAPE는 참고용 -- 극단치에 민감):")
    print(summary_by_h.round(3))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for col in acc_h["variable"].unique():
        sub = acc_h[acc_h["variable"] == col].sort_values("horizon")
        axes[0].plot(sub["horizon"], sub["r2"], marker="o", markersize=3, linewidth=0.8, alpha=0.35, color="tab:blue")
        axes[1].plot(sub["horizon"], sub["mae"], marker="o", markersize=3, linewidth=0.8, alpha=0.35, color="tab:red")
    axes[0].plot(summary_by_h.index, summary_by_h["r2"], marker="o", markersize=8, linewidth=2.5, color="black", label="평균")
    axes[1].plot(summary_by_h.index, summary_by_h["mae"], marker="o", markersize=8, linewidth=2.5, color="black", label="평균")
    axes[0].set_title(f"[{eid}] horizon별 R² (변수별 가는 선 + 전체 평균 굵은 선)")
    axes[0].set_xlabel("horizon (몇 시간 뒤 예측)")
    axes[0].legend()
    axes[1].set_title(f"[{eid}] horizon별 MAE")
    axes[1].set_xlabel("horizon (몇 시간 뒤 예측)")
    axes[1].legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# ===================== [실험] 결과 저장 =====================
for eid, r in multistep_results.items():
    df = r["all_results"].copy()
    df["variable"] = df["variable"].str.replace("\n", " ")
    path_out = f"chronos_multistep_results_{eid}.csv"
    df.to_csv(path_out, index=False, encoding="utf-8-sig")
    print(f"저장 완료: {path_out} ({len(df)}행)")


# Chronos-2 Full 파인튜닝: GitHub review_v2 재현

- 예측 길이: **1시간** (`prediction_length=1`)
- 대상: 품질 변수와 `POLYCOM 운전시간`을 제외한 공정 변수 38개
- 입력 보조 변수: `POLYCOM 운전시간`은 past covariate로만 사용
- 분할: 2CM·3CM·4CM 각각 시간순 **70% 학습 / 15% 검증 / 15% 테스트**
- 검증: 각 설비의 검증 기간 전반에서 최대 128개 윈도우를 결정론적으로 선택
- 파인튜닝: `full`, learning rate `1e-6`, 1,000 steps, batch size 64, seed 42
- 테스트 구간은 학습과 모델 선택에 사용하지 않고 마지막 비교에만 사용

Full 파인튜닝은 시간이 오래 걸리므로 아래 셀을 순서대로 직접 실행합니다. 출력 폴더가 이미 비어 있지
않으면 기존 체크포인트를 덮어쓰지 않고 오류를 냅니다.


In [ ]:
# ===================== Full 파인튜닝 CONFIG =====================
FT_TRAIN_FRAC = 0.70
FT_VAL_FRAC = 0.15
FT_TEST_FRAC = 0.15
FT_CONTEXT_LENGTH = CONTEXT_LENGTH
FT_PREDICTION_LENGTH = PREDICTION_LENGTH
FT_VALIDATION_WINDOWS_PER_EQUIPMENT = 128
FT_NUM_STEPS = 1000
FT_BATCH_SIZE = 64
FT_LEARNING_RATE = 1e-6
FT_MODE = "full"
FT_SEED = 42
FT_OUTPUT_DIR = Path("checkpoints") / PROTOCOL_VERSION / "all_process_full_pred1"
FINETUNED_MODEL_DIR = FT_OUTPUT_DIR / "final"

assert FT_PREDICTION_LENGTH == 1
print(f"mode={FT_MODE}, pred_len={FT_PREDICTION_LENGTH}, context={FT_CONTEXT_LENGTH}, "
      f"steps={FT_NUM_STEPS}, batch={FT_BATCH_SIZE}, lr={FT_LEARNING_RATE}, seed={FT_SEED}")


In [ ]:
# ===================== 시간순 70/15/15 분할 + 검증 윈도우 구성 =====================
if PREPROCESSED_CSV.exists():
    ft_full_df = pd.read_csv(PREPROCESSED_CSV, parse_dates=["timestamp"])
else:
    _frames = []
    for _eid in EQUIPMENT_IDS:
        _data, _time, _targets, _, _ = load_data(_eid)
        _frames.append(equipment_frame(_eid, _data, _time, _targets))
    ft_full_df = pd.concat(_frames, ignore_index=True)
    PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    ft_full_df.to_csv(PREPROCESSED_CSV, index=False, encoding="utf-8-sig")

ft_variable_cols = [c for c in ft_full_df.columns if c not in ["item_id", "timestamp", DOWNTIME_COL]]
assert not set(QUALITY_COLS).intersection(ft_variable_cols)
assert DOWNTIME_COL not in ft_variable_cols

ft_train_df, ft_val_df, ft_test_df = chronological_split(
    ft_full_df, train_frac=FT_TRAIN_FRAC, val_frac=FT_VAL_FRAC,
)
ft_val_windows_df, ft_validation_manifest = validation_windows(
    ft_train_df, ft_val_df, ft_variable_cols,
    prediction_length=FT_PREDICTION_LENGTH, context_length=FT_CONTEXT_LENGTH,
    windows_per_item=FT_VALIDATION_WINDOWS_PER_EQUIPMENT,
)
train_inputs = to_chronos_inputs(ft_train_df, ft_variable_cols, FT_PREDICTION_LENGTH)
val_inputs = to_chronos_inputs(ft_val_windows_df, ft_variable_cols, FT_PREDICTION_LENGTH)
for _window in val_inputs:
    _labels = _window["context"][:_window["n_targets"], -FT_PREDICTION_LENGTH:]
    if not torch.isfinite(_labels).any().item():
        raise ValueError("검증 윈도우의 예측 구간에 유효한 target이 없습니다.")

print(pd.DataFrame({
    "train": ft_train_df.groupby("item_id").size(),
    "validation": ft_val_df.groupby("item_id").size(),
    "test": ft_test_df.groupby("item_id").size(),
}))
print(ft_validation_manifest.groupby("item_id").agg(
    windows=("window_id", "size"), observed_labels=("observed_labels", "sum"),
    first_forecast=("forecast_start", "min"), last_forecast=("forecast_end", "max"),
))
print(f"targets={len(ft_variable_cols)}, train_inputs={len(train_inputs)}, val_inputs={len(val_inputs)}")


In [ ]:
# ===================== Full 파인튜닝 실행 + 재현 정보 저장 =====================
import json
import time
from transformers import set_seed

if FT_OUTPUT_DIR.exists() and any(FT_OUTPUT_DIR.iterdir()):
    raise FileExistsError(f"기존 실행을 덮어쓰지 않습니다: {FT_OUTPUT_DIR}")
FT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ft_validation_manifest.to_csv(FT_OUTPUT_DIR / "validation_windows.csv", index=False)

ft_metadata = {
    "protocol": PROTOCOL_VERSION,
    "status": "started",
    "model_id": MODEL_ID,
    "prediction_length": FT_PREDICTION_LENGTH,
    "context_length": FT_CONTEXT_LENGTH,
    "target_columns": ft_variable_cols,
    "past_covariates": [DOWNTIME_COL],
    "split": {"train": FT_TRAIN_FRAC, "validation": FT_VAL_FRAC, "test": FT_TEST_FRAC},
    "finetune_mode": FT_MODE,
    "learning_rate": FT_LEARNING_RATE,
    "num_steps": FT_NUM_STEPS,
    "batch_size": FT_BATCH_SIZE,
    "seed": FT_SEED,
    "source_xlsx_sha256": file_sha256(EXCEL_PATH),
    "processed_csv_sha256": file_sha256(PREPROCESSED_CSV),
    "versions": runtime_versions(),
    "validation_windows": len(ft_validation_manifest),
    "validation_observed_labels": int(ft_validation_manifest["observed_labels"].sum()),
}
(FT_OUTPUT_DIR / "run_metadata.json").write_text(
    json.dumps(ft_metadata, ensure_ascii=False, indent=2), encoding="utf-8",
)

set_seed(FT_SEED)
loss_callback = make_loss_history_callback()
_ft_start = time.time()
pipeline_ft = pipeline.fit(
    inputs=train_inputs,
    prediction_length=FT_PREDICTION_LENGTH,
    validation_inputs=val_inputs,
    finetune_mode=FT_MODE,
    learning_rate=FT_LEARNING_RATE,
    num_steps=FT_NUM_STEPS,
    batch_size=FT_BATCH_SIZE,
    context_length=FT_CONTEXT_LENGTH,
    output_dir=FT_OUTPUT_DIR,
    finetuned_ckpt_name="finetuned-ckpt",
    callbacks=[loss_callback],
    seed=FT_SEED,
    data_seed=FT_SEED,
)
_ft_elapsed = time.time() - _ft_start
pipeline_ft.save_pretrained(FINETUNED_MODEL_DIR)
loss_history = loss_history_from_callback(loss_callback)
loss_history.to_csv(FT_OUTPUT_DIR / "loss_history.csv", index=False)
if loss_history["val_loss"].dropna().empty:
    raise RuntimeError("유효한 validation loss가 기록되지 않아 실행을 완료로 표시하지 않습니다.")
ft_metadata["status"] = "complete"
ft_metadata["elapsed_minutes"] = _ft_elapsed / 60
(FT_OUTPUT_DIR / "run_metadata.json").write_text(
    json.dumps(ft_metadata, ensure_ascii=False, indent=2), encoding="utf-8",
)
print(f"Full 파인튜닝 완료: {FINETUNED_MODEL_DIR} ({_ft_elapsed/60:.1f}분)")


In [ ]:
# ===================== 손대지 않은 테스트 15%에서 zero-shot과 Full 모델 재예측 =====================
def evaluate_on_test(model):
    output = []
    for equipment_id in EQUIPMENT_IDS:
        full_group = ft_full_df.loc[ft_full_df["item_id"].eq(equipment_id)].sort_values("timestamp").reset_index(drop=True)
        train_group = ft_train_df.loc[ft_train_df["item_id"].eq(equipment_id)]
        test_group = ft_test_df.loc[ft_test_df["item_id"].eq(equipment_id)]
        test_start = test_group["timestamp"].min()
        test_start_idx = int(np.flatnonzero(full_group["timestamp"].ge(test_start).to_numpy())[0])
        slice_start = max(0, test_start_idx - FT_CONTEXT_LENGTH)
        eval_group = full_group.iloc[slice_start:].reset_index(drop=True)
        eval_data = eval_group[[*ft_variable_cols, DOWNTIME_COL]].copy()
        is_down = eval_data[DOWNTIME_COL].eq(0).fillna(False).to_numpy()

        train_running = ~train_group[DOWNTIME_COL].eq(0).fillna(False)
        variable_scale = train_group.loc[train_running, ft_variable_cols].std().fillna(0.0).to_numpy()
        result = rolling_forecast_review_v2(
            model, eval_data, ft_variable_cols, variable_scale, is_down,
            context_length=FT_CONTEXT_LENGTH, prediction_length=FT_PREDICTION_LENGTH,
            stride=STRIDE, batch_windows=BATCH_WINDOWS,
            quantile_low=ANOMALY_QUANTILE_LOW, quantile_high=ANOMALY_QUANTILE_HIGH,
            severity_eps=SEVERITY_EPS,
        )
        result["timestamp"] = [eval_group["timestamp"].iloc[i] for i in result["idx"]]
        result = result.loc[result["timestamp"].ge(test_start)].copy()
        result.insert(0, "equipment_id", equipment_id)
        output.append(result)
    return pd.concat(output, ignore_index=True)

all_results_zeroshot_test = evaluate_on_test(pipeline)
all_results_ft = evaluate_on_test(pipeline_ft)
print(f"zero-shot test rows={len(all_results_zeroshot_test):,}, full test rows={len(all_results_ft):,}")


In [ ]:
# ===================== 동일한 테스트 15%에서 zero-shot vs Full 비교 =====================
def accuracy_by_equipment(result_df, model_name):
    parts = []
    for equipment_id, group in result_df.groupby("equipment_id", sort=False):
        acc = compute_accuracy_summary(group)
        acc.insert(0, "equipment_id", equipment_id)
        acc.insert(1, "model", model_name)
        parts.append(acc)
    return pd.concat(parts, ignore_index=True)

accuracy_summary_zeroshot_test = accuracy_by_equipment(all_results_zeroshot_test, "zero-shot")
accuracy_summary_ft = accuracy_by_equipment(all_results_ft, "full-finetuned")
metric_cols = ["mae", "rmse", "mape", "mae_vs_naive", "r2", "coverage", "n_scored"]
compare = accuracy_summary_zeroshot_test[["equipment_id", "variable", "label", *metric_cols]].merge(
    accuracy_summary_ft[["equipment_id", "variable", *metric_cols]],
    on=["equipment_id", "variable"], suffixes=("_zeroshot", "_finetuned"),
)

print("=== 테스트 구간 전체 변수 평균: zero-shot -> Full 파인튜닝 ===")
print(f"MAE/naive  : {compare['mae_vs_naive_zeroshot'].mean():.3f} -> {compare['mae_vs_naive_finetuned'].mean():.3f}")
print(f"R²         : {compare['r2_zeroshot'].mean():.3f} -> {compare['r2_finetuned'].mean():.3f}")
print(f"coverage(%) : {compare['coverage_zeroshot'].mean():.2f} -> {compare['coverage_finetuned'].mean():.2f}")
print(f"naive보다 나은 설비-변수: {int((compare['mae_vs_naive_zeroshot'] < 1).sum())}/{len(compare)} -> "
      f"{int((compare['mae_vs_naive_finetuned'] < 1).sum())}/{len(compare)}")

all_results_zeroshot_test.to_csv(FT_OUTPUT_DIR / "test_predictions_zeroshot.csv", index=False, encoding="utf-8-sig")
all_results_ft.to_csv(FT_OUTPUT_DIR / "test_predictions_full.csv", index=False, encoding="utf-8-sig")
compare.to_csv(FT_OUTPUT_DIR / "test_metrics_comparison.csv", index=False, encoding="utf-8-sig")

comp_plot = compare.assign(key=lambda frame: frame["equipment_id"] + " | " + frame["label"])
comp_plot = comp_plot.sort_values("mae_vs_naive_finetuned")
fig, ax = plt.subplots(figsize=(11, max(7, len(comp_plot) * 0.16)))
y = np.arange(len(comp_plot))
ax.barh(y - 0.2, comp_plot["mae_vs_naive_zeroshot"], height=0.4, color="tab:gray", label="zero-shot")
ax.barh(y + 0.2, comp_plot["mae_vs_naive_finetuned"], height=0.4, color="tab:blue", label="Full 파인튜닝")
ax.axvline(1.0, color="black", linewidth=1, linestyle="--")
ax.set_yticks(y)
ax.set_yticklabels(comp_plot["key"], fontsize=6)
ax.invert_yaxis()
ax.set_title("테스트 15% MAE / naive: zero-shot vs Full 파인튜닝 (<1이 좋음)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

compare.sort_values(["equipment_id", "mae_vs_naive_finetuned"])
